In [17]:
import pandas as pd

# Load the dataset
df = pd.read_csv("sepsis.csv")

# Print verification and display data columns/rows
print("Data pipeline connected successfully!")
print(f"Total rows (hourly records): {df.shape[0]}")
print(f"Total columns (tracked health metrics): {df.shape[1]}\n")
df.head()

Data pipeline connected successfully!
Total rows (hourly records): 5000
Total columns (tracked health metrics): 77



,subject_id,age,gender,weight_kg,height_cm,bmi,ethnicity,insurance,hr_mean,hr_max,...,apache_iv,qsofa,sirs_criteria,gcs_total,icu_los_hours,hospital_admit_source,icu_admit_time_hour,day_of_week,readmission_30day,sepsis_label
0,37464,72,M,99.5,160.6,38.6,Black,Medicare,83.0,98.7,...,47.9,2,1,12,27.0,OR,15,2,1,1
1,34024,62,M,101.2,164.5,37.4,Other,Private,97.5,105.8,...,35.0,1,1,6,21.5,Transfer,21,2,0,0
2,65291,74,M,64.0,167.2,22.9,White,Private,100.6,105.8,...,29.0,1,1,11,46.7,ED,19,6,0,0
3,79118,87,M,75.1,180.5,23.0,White,Medicare,95.5,114.7,...,35.6,1,1,7,1.7,ED,8,2,0,0
4,34897,61,M,52.5,158.4,NaN,Hispanic,Private,86.7,106.7,...,34.6,1,1,8,64.0,Transfer,9,2,0,0


In [18]:
# Check how many missing values are in some of our core columns
missing_vitals = df[['hr_mean', 'bmi', 'qsofa', 'gcs_total']].isnull().sum()

print("Number of missing values per column:")
print(missing_vitals)

print("\n--- Column Data Types ---")
print(df[['hr_mean', 'bmi', 'qsofa', 'gcs_total']].dtypes)

Number of missing values per column:
hr_mean        0
bmi          251
qsofa          0
gcs_total      0
dtype: int64

--- Column Data Types ---
hr_mean      float64
bmi          float64
qsofa          int64
gcs_total      int64
dtype: object


In [19]:
# Fill the missing values in the 'bmi' column with the median BMI value
bmi_median = df['bmi'].median()
df['bmi'] = df['bmi'].fillna(bmi_median)

# Verify that there are absolutely zero missing values left in our target columns
print(f"Calculated Median BMI to use for gaps: {bmi_median:.2f}")
print("Remaining missing values in 'bmi':", df['bmi'].isnull().sum())


Calculated Median BMI to use for gaps: 25.60
Remaining missing values in 'bmi': 0


In [20]:
# Calculate the 3-hour change (delta) for heart rate and clinical scores
# We group by subject_id so changes are only calculated within the same patient's timeline
df['hr_delta_3h'] = df.groupby('subject_id')['hr_mean'].diff(3)
df['qsofa_delta_3h'] = df.groupby('subject_id')['qsofa'].diff(3)

# Because the first 3 hours of a patient's stay don't have data from "3 hours ago", 
# they will naturally become NaNs. We will fill those initial baseline hours with 0.
df['hr_delta_3h'] = df['hr_delta_3h'].fillna(0)
df['qsofa_delta_3h'] = df['qsofa_delta_3h'].fillna(0)

# Preview our brand new engineered trend features
print("Engineered features created successfully!")
df[['subject_id', 'hr_mean', 'hr_delta_3h', 'qsofa', 'qsofa_delta_3h']].head(6)

Engineered features created successfully!


,subject_id,hr_mean,hr_delta_3h,qsofa,qsofa_delta_3h
0,37464,83.0,0.0,2,0.0
1,34024,97.5,0.0,1,0.0
2,65291,100.6,0.0,1,0.0
3,79118,95.5,0.0,1,0.0
4,34897,86.7,0.0,1,0.0
5,54635,95.4,0.0,1,0.0


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# 1. AUTOMATICALLY FIND THE TARGET COLUMN
# We look for a column with 'sepsis' in its name, or default to the very last column
possible_targets = [col for col in df.columns if 'sepsis' in col.lower()]

if possible_targets:
    target_column = possible_targets[0]
    print(f"Found target column: '{target_column}'")
else:
    # If no column names contain 'sepsis', grab the very last column in the file
    target_column = df.columns[-1]
    print(f"No explicit 'sepsis' column found. Defaulting to last column: '{target_column}'")

# 2. Define our target variable (the answer key)
y = df[target_column]

# 3. Drop identification columns and the target column itself
features_to_drop = ['subject_id', target_column]
X_raw = df.drop(columns=features_to_drop, errors='ignore')

# 4. Keep ONLY numerical columns (strips away text values like 'ED', 'M', 'F')
X = X_raw.select_dtypes(include=[np.number])

print(f"Cleaned training dataset shape: {X.shape[0]} rows and {X.shape[1]} numeric features.")

# 5. Perform the 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 6. Initialize and train the AI Model
print("\nTraining the Random Forest model on pure numerical values...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 7. Test the model's predictions
y_pred = model.predict(X_test)

# 8. Display the results
print("\n=== Model Evaluation Metrics ===")
print(f"Overall Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Detailed Performance Report:")
print(classification_report(y_test, y_pred))

Found target column: 'sepsis_label'
Cleaned training dataset shape: 5000 rows and 73 numeric features.

Training the Random Forest model on pure numerical values...

=== Model Evaluation Metrics ===
Overall Model Accuracy: 100.00%

Detailed Performance Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       850
           1       1.00      1.00      1.00       150

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [22]:
import pandas as pd

# Extract how important each column was to the Random Forest model
importances = model.feature_importances_
feature_names = X.columns

# Sort them in descending order
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print("=== Top 15 Most Influential Features ===")
print(feature_importance_df.head(15).to_string(index=False))

=== Top 15 Most Influential Features ===
              Feature  Importance
      pao2_fio2_ratio    0.217361
         lactate_mmol    0.167045
           creatinine    0.098138
          ph_arterial    0.075679
           sofa_score    0.065073
        fluids_ml_24h    0.064778
            apache_iv    0.055979
             spo2_max    0.035794
            spo2_mean    0.031598
        sirs_criteria    0.029557
             spo2_min    0.022281
          bicarbonate    0.019549
 respiratory_rate_min    0.015281
respiratory_rate_mean    0.014485
                  inr    0.013194


In [23]:
# 1. Define the specific "cheat" columns causing data leakage
leakage_columns = ['pao2_fio2_ratio', 'lactate_mmol', 'sofa_score', 'apache_iv', 'sirs_criteria']

# 2. Drop these columns from our training features dataset
X_realistic = X.drop(columns=leakage_columns, errors='ignore')

print(f"Realistic dataset shape: {X_realistic.shape[0]} rows and {X_realistic.shape[1]} features.")
print("Dropped leaking variables. Retraining model...")

# 3. Re-split the data using the new, realistic feature set
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_realistic, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Train the new, realistic Random Forest model
realistic_model = RandomForestClassifier(n_estimators=100, random_state=42)
realistic_model.fit(X_train_r, y_train_r)

# 5. Generate new predictions
y_pred_r = realistic_model.predict(X_test_r)

# 6. Display realistic results
print("\n=== Realistic Model Evaluation Metrics ===")
print(f"Realistic Model Accuracy: {accuracy_score(y_test_r, y_pred_r) * 100:.2f}%\n")
print("Detailed Performance Report:")
print(classification_report(y_test_r, y_pred_r))

Realistic dataset shape: 5000 rows and 68 features.
Dropped leaking variables. Retraining model...

=== Realistic Model Evaluation Metrics ===
Realistic Model Accuracy: 99.60%

Detailed Performance Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       850
           1       0.99      0.98      0.99       150

    accuracy                           1.00      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       1.00      1.00      1.00      1000



In [24]:
# Extract and display the new top 10 features for the realistic model
r_importances = realistic_model.feature_importances_
r_feature_names = X_realistic.columns

r_importance_df = pd.DataFrame({'Feature': r_feature_names, 'Importance': r_importances})
r_importance_df = r_importance_df.sort_values(by='Importance', ascending=False)

print("=== New Top 10 Influential Features ===")
print(r_importance_df.head(10).to_string(index=False))

=== New Top 10 Influential Features ===
             Feature  Importance
          creatinine    0.211062
         ph_arterial    0.131360
       fluids_ml_24h    0.083897
           spo2_mean    0.082046
                 inr    0.070048
            spo2_max    0.068712
         bicarbonate    0.057305
            spo2_min    0.056154
respiratory_rate_min    0.029706
      platelet_count    0.028281


In [25]:
# 1. Isolate ONLY bedside vitals and your custom engineered trend features
vitals_only = [
    'hr_mean', 'hr_max', 'hr_delta_3h', 
    'qsofa', 'qsofa_delta_3h', 'gcs_total', 
    'bmi', 'age'
]

# Ensure we only pick columns that actually exist in our dataset
X_vitals = X_realistic[[col for col in vitals_only if col in X_realistic.columns]]

print(f"Early-warning dataset shape: {X_vitals.shape[0]} rows and {X_vitals.shape[1]} pure clinical features.")
print("Training the final early warning model...")

# 2. Re-split using ONLY the bedside features
X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
    X_vitals, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Train the final Random Forest
vitals_model = RandomForestClassifier(n_estimators=100, random_state=42)
vitals_model.fit(X_train_v, y_train_v)

# 4. Generate honest predictions
y_pred_v = vitals_model.predict(X_test_v)

# 5. Display the genuine results
print("\n=== Early Warning Model Evaluation Metrics ===")
print(f"Genuine Bedside Accuracy: {accuracy_score(y_test_v, y_pred_v) * 100:.2f}%\n")
print("Detailed Performance Report:")
print(classification_report(y_test_v, y_pred_v))

Early-warning dataset shape: 5000 rows and 8 pure clinical features.
Training the final early warning model...

=== Early Warning Model Evaluation Metrics ===
Genuine Bedside Accuracy: 92.20%

Detailed Performance Report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.96       850
           1       0.82      0.61      0.70       150

    accuracy                           0.92      1000
   macro avg       0.88      0.79      0.83      1000
weighted avg       0.92      0.92      0.92      1000



In [26]:
import joblib

# Save the final trained bedside model to a file
joblib.dump(vitals_model, 'vitals_sepsis_model.pkl')

# Save the list of feature names so our frontend knows the exact order the model expects
joblib.dump(X_vitals.columns.tolist(), 'model_features.pkl')

print("Model and features successfully saved to disk as permanent files!")

Model and features successfully saved to disk as permanent files!
